In [3]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Document
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from dotenv import load_dotenv
load_dotenv()

# connect to Qdrant Cloud
client = QdrantClient(
    url=os.getenv("QDRANT_API_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    cloud_inference=True
)

In [3]:
# # create collection
# client.create_collection(
#     collection_name="leo-docs",
#     vectors_config=VectorParams(size=384, distance=Distance.COSINE),
# )

In [2]:
class EmbeddingDocument:
    def __init__(self, path):
        self.path = path
        self.name = os.path.basename(path)
        self.text = self._load_text()

    def _load_text(self):
        with open(self.path, 'r', encoding='utf-8') as f:
            return f.read()
        
    def chunk_text_plain(self):
        """
        Plain chunking: the entire file content is a single chunk.
        """
        loader = TextLoader(self.path, encoding='utf-8')
        data = loader.load()

        # data is a list of Document objects from the loader (usually just one)
        for doc in data:
            yield (self.name, doc.page_content)
            

    def chunk_text(self, chunk_size=1000, chunk_overlap=100):
        """
        Recursive chunking using LangChain's RecursiveCharacterTextSplitter.
        """
        loader = TextLoader(self.path, encoding='utf-8')
        data = loader.load()

        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            add_start_index=True,
        )
        all_splits = text_splitter.split_documents(data)

        # Yield the (name, text) shape chunk_text
        for split in all_splits:
            yield (self.name, split.page_content)

In [3]:
chunks = []
for file_path in os.listdir("docs"):
    if file_path.endswith(".txt"):
        doc = EmbeddingDocument(os.path.join("docs", file_path))
        doc_chunks = list(doc.chunk_text_plain())
        chunks = chunks + doc_chunks


In [4]:
chunks

[('05_soft_skills.txt',
  "Here is a look at Leonardo Acquaroli's soft skills and how they developed.\n\nTEAM PLAYER\nLeonardo considers himself a genuine team player. This shows up throughout his career: from co-founding Loomy and UnimAI with partners, to collaborating within the AI Applications team at Banco BPM, which operates on a Hub & Spoke consultancy model requiring close coordination with other teams across the bank.\n\nNATURAL LEADERSHIP\nLeonardo's leadership emerged naturally through many hackathons and lab experiences, and he has deliberately trained it further by studying many leadership-related books and essays. This natural leadership is reflected in his track record: he was Vice President of the Starting Finance Club Polimarche, Co-founder & Director of UnimAI, and he has led numerous events for student associations. He also placed well in several technical hackathons, including a victory at the Sky Data Hackathon.\n\nCURIOSITY\nLeonardo would describe himself as (too 

In [6]:
# points generator
points = []
for i, chunk in enumerate(chunks):
    point = PointStruct(
        id=i,
        vector=Document(
            text=f"<{chunk[0]}>\n\n{chunk[1]}",
            model="sentence-transformers/all-MiniLM-L6-v2"
        ),
        payload={
            "doc_name": chunk[0],
            "text": chunk[1]
        }
    )
    points.append(point)

# upsert points to collection
client.upsert(
  collection_name="leo-docs",
  points=points,
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [4]:
# generate query embedding
query_text = "hard skills"

# search for similar menu items
results = client.query_points(
    collection_name="leo-docs",
    query=Document(text=query_text, model="sentence-transformers/all-MiniLM-L6-v2"),
    with_payload=True,
    limit=3
)

# print results
for result in results.points:
    print(f"Item: {result.payload.get('doc_name', 'N/A')}")
    print(f"Score: {result.score}")
    print(f"Description: {result.payload['text'][:150]}...")
    print("---")

Item: 05_soft_skills.txt
Score: 0.31778562
Description: Here is a look at Leonardo Acquaroli's soft skills and how they developed.

TEAM PLAYER
Leonardo considers himself a genuine team player. This shows u...
---
Item: 04_hard_skills.txt
Score: 0.30226955
Description: Here is an overview of Leonardo Acquaroli's technical and hard skills.

GENERAL TECHNICAL SKILLS
Leonardo is proficient in Python and R for data analy...
---
Item: 06_competitions.txt
Score: 0.28158394
Description: Here is Leonardo Acquaroli's track record in business competitions and technical hackathons.

BUSINESS COMPETITIONS
Leonardo won iDays Milano in 2022 ...
---
